# Kansas environments: build the app layers

Builds the files in `data/` that the web app (`index.html`) reads, from the 1-km rasters in `inputs/`, and runs the default zoning once in Python as a check (`figures/zones_default.png`).

Zoning is hierarchical, following Sexton-Bowser, Roozeboom and Patrignani (2026, *Agric. For. Meteorol.* 378, 110999). First, k-means on climate layers gives *macrozones*; then each macrozone is split by its own k-means on soil layers into *microzones*, so every microzone sits inside one macrozone. Every layer is z-scored over the whole state, so each counts equally. The silhouette score guides the number of zones. The paper also applied PCA before k-means; it is left out here because keeping 95 % of the variance is almost a pure rotation of the data, so the zones barely change.

| layer | units | input file | default step |
|---|---|---|---|
| Annual rainfall | mm | `ppt_annual_normal.tif` (PRISM 1991–2020 normal) | macro |
| Annual reference ET (ETo) | mm | `eto_annual_normal.tif` (gridMET 1991–2020 mean) | macro |
| Aridity index | rainfall / ETo | computed | off |
| Atmospheric water deficit | mm, ETo − rainfall | computed | off |
| Growing degree days | °C·d | `gdd10_annual_normal.tif` (PRISM daily 1991–2020, base 10 °C, Tmax capped at 30 °C, Tmin floored at 10 °C) | macro |
| Deficit-days | FAW-days per year | `deficit_days_median_2002_2025.tif` (Kansas 1-km soil moisture model) | off |
| Surplus-days | FAW-days per year | `surplus_days_median_2002_2025.tif` (Kansas 1-km soil moisture model) | off |
| Plant available water capacity | mm, 0–60 cm | `pawc_0_60cm.tif` (−10 kPa to wilting point) | micro |
| Soil organic matter | kg m⁻², whole profile | `som.tif` (SSURGO, CSRL) | micro |
| Clay | %, 0–60 cm | `clay_0_60cm.tif` (SSURGO, CSRL) | micro |
| Sand | %, 0–60 cm | `sand_0_60cm.tif` (SSURGO, CSRL) | off |

All inputs are on the same 1-km grid (EPSG:4326, 336 × 832) as `kansas_mask.tif`. To update a layer, replace its file and rerun this notebook.

In [ ]:
import os, json, gzip
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'python' else os.getcwd()   # run from python/ or the repository root
INP, DATA, FIGS = (os.path.join(ROOT, d) for d in ('inputs', 'data', 'figures'))
os.makedirs(DATA, exist_ok=True); os.makedirs(FIGS, exist_ok=True)
import numpy as np, pandas as pd, rasterio, geopandas as gpd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': ['Arial', 'DejaVu Sans'], 'font.size': 9, 'axes.linewidth': 0.8})
with rasterio.open(os.path.join(INP, 'kansas_mask.tif')) as src:
    shape, transform = src.shape, src.transform
    valid = src.read(1) == 1
print(ROOT); print(shape, valid.sum(), 'pixels')

## Growing degree days (Earth Engine, only if the input is missing)
Annual sums from PRISM daily (4 km) for 1991–2020, one Earth Engine request per year (a single request for all 30 years times out), cached in `GDD_CACHE`, averaged, and snapped to the grid with bilinear resampling. Needs an Earth Engine project (`EE_PROJECT`).

In [ ]:
fp = os.path.join(INP, 'gdd10_annual_normal.tif')
EE_PROJECT = 'soilmoisture-1490511638539'
GDD_CACHE = r'D:\kansas_1km_cache\prism_gdd10'
if os.path.isfile(fp):
    print('File exists:', fp)
else:
    import ee, requests, time, rioxarray as rxr, xarray as xr
    from rasterio.enums import Resampling
    ee.Initialize(project=EE_PROJECT)
    os.makedirs(GDD_CACHE, exist_ok=True)
    region = ee.FeatureCollection('TIGER/2018/States').filter(ee.Filter.eq('NAME', 'Kansas'))
    def gdd(img):      # modified 10/30 method (the 50/86 °F corn method)
        tmax = img.select('tmax').min(30).max(10); tmin = img.select('tmin').max(10).min(30)
        return tmax.add(tmin).divide(2).subtract(10).rename('gdd')
    for y in range(1991, 2021):
        out = os.path.join(GDD_CACHE, f'gdd10_{y}.tif')
        if os.path.isfile(out): continue
        img = ee.ImageCollection('OREGONSTATE/PRISM/AN81d').filterDate(f'{y}-01-01', f'{y + 1}-01-01').map(gdd).sum()
        url = img.clip(region.geometry().buffer(10000)).getDownloadUrl({'region': region.geometry().buffer(10000), 'scale': 4000, 'crs': 'EPSG:4326', 'format': 'GEO_TIFF'})
        for k in range(3):
            r = requests.get(url)
            if r.content[:2] in (b'II', b'MM'): break
            time.sleep(5)
        assert r.content[:2] in (b'II', b'MM'), r.content[:200]
        with open(out, 'wb') as fh: fh.write(r.content)
        print(y, end=' ', flush=True)
    stack = xr.concat([rxr.open_rasterio(os.path.join(GDD_CACHE, f'gdd10_{y}.tif'), masked=True).squeeze() for y in range(1991, 2021)], 'year').mean('year')
    template = rxr.open_rasterio(os.path.join(INP, 'kansas_mask.tif')).squeeze()
    da = stack.rio.write_crs('EPSG:4326').rio.reproject_match(template, resampling=Resampling.bilinear, nodata=np.nan)
    da.where(template.values == 1).astype('float32').rio.to_raster(fp, compress='deflate')
    print('\n' + fp)

## Layer stack

In [ ]:
def read(name):
    with rasterio.open(os.path.join(INP, name)) as src: a = src.read(1).astype('float64')
    return a[valid]

ppt, eto = read('ppt_annual_normal.tif'), read('eto_annual_normal.tif')
L = [  # key, name, units, default step, short source, values on Kansas pixels
    ('ppt',  'Annual rainfall',          'mm',          'macro', 'PRISM 1991-2020 normal',              ppt),
    ('eto',  'Annual reference ET',      'mm',          'macro', 'gridMET 1991-2020 mean',              eto),
    ('ai',   'Aridity index',            'P/ETo',       'off',   'annual rainfall / annual ETo',        ppt / eto),
    ('awd',  'Atmospheric water deficit', 'mm',         'off',   'annual ETo - annual rainfall',        eto - ppt),
    ('gdd',  'Growing degree days',      '°C·d',        'macro', 'PRISM daily 1991-2020, base 10 °C',   read('gdd10_annual_normal.tif')),
    ('def',  'Deficit-days',             'FAW-days',    'off',   'median 2002-2025',                    read('deficit_days_median_2002_2025.tif')),
    ('sur',  'Surplus-days',             'FAW-days',    'off',   'median 2002-2025',                    read('surplus_days_median_2002_2025.tif')),
    ('pawc', 'Plant available water capacity', 'mm',    'micro', '0-60 cm, -10 kPa to wilting point',   read('pawc_0_60cm.tif')),
    ('som',  'Soil organic matter',      'kg/m²',       'micro', 'SSURGO, whole profile',                read('som.tif')),
    ('clay', 'Clay',                     '%',           'micro', 'SSURGO, 0-60 cm',                      read('clay_0_60cm.tif')),
    ('sand', 'Sand',                     '%',           'off',   'SSURGO, 0-60 cm',                      read('sand_0_60cm.tif')),
]
for k, n, u, s, src, v in L:
    bad = ~np.isfinite(v)
    if bad.any(): v[bad] = np.nanmedian(v)            # fill any gaps with the state median
    print(f'{n:32s} {np.min(v):8.1f} {np.median(v):8.1f} {np.max(v):8.1f} {u:10s} gaps filled: {bad.sum()}')
X = np.column_stack([v for *_, v in L])
keys = [k for k, *_ in L]
pd.DataFrame(np.corrcoef(X.T), index=keys, columns=keys).round(2)

## Reference run (Python)

The app's defaults. Macro: rainfall, ETo and GDD. Micro: PAWC, organic matter and clay. Each step is z-scored statewide, then k-means runs on the z-scores (k-means++, 5 starts).

Silhouette scores are computed for 2–8 macrozones and 2–5 microzones. Each candidate is fitted on a 20,000-pixel sample and scored on 3,000 pixels of it. The macro curve is nearly flat (k = 2 and 3 within about 0.02), because Kansas climate is a smooth gradient. So the defaults are the paper's result, 3 macrozones with 2 microzones each.

Zones are numbered by the mean of the first layer of the step: macro by rainfall, dry to wet; micro by PAWC, low to high. The app runs the same steps in JavaScript.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import colorsys
MACRO, MICRO = ['ppt', 'eto', 'gdd'], ['pawc', 'som', 'clay']
rs = np.random.default_rng(0)
zs = lambda cols: (X[:, cols] - X[:, cols].mean(0)) / X[:, cols].std(0)

def sil_curve(A, ks):
    s = rs.choice(len(A), min(20000, len(A)), replace=False)
    return {k: silhouette_score(A[s], KMeans(k, n_init=5, random_state=0).fit_predict(A[s]), sample_size=min(3000, len(s)), random_state=0) for k in ks}

def kmeans(A, k, order_col):
    lab = KMeans(k, n_init=5, random_state=0).fit_predict(A)
    o = np.argsort([order_col[lab == j].mean() for j in range(k)])
    return np.argsort(o)[lab]                                   # 0 = lowest mean of order_col

ci, si = [keys.index(k) for k in MACRO], [keys.index(k) for k in MICRO]
Pc, Ps = zs(ci), zs(si)
K1, K2 = 3, 2
print('macro silhouette:', {k: round(v, 2) for k, v in sil_curve(Pc, range(2, 9)).items()})
macro = kmeans(Pc, K1, X[:, ci[0]])
micro = np.zeros_like(macro)
for m in range(K1):
    i = macro == m
    print(f'  macrozone {m + 1}: micro silhouette', {k: round(v, 2) for k, v in sil_curve(Ps[i], range(2, 6)).items()})
    micro[i] = kmeans(Ps[i], K2, X[i, si[0]])
code_ = (macro + 1) * 10 + micro + 1

HUES = [25, 54, 137, 209, 270, 331, 173, 0]                     # degrees, same colours as the app
def colour(m, u, k2):
    l = 78 - 44 * u / (k2 - 1) if k2 > 1 else 55; s = 45 + 0.4 * (80 - l) if k2 > 1 else 55
    return colorsys.hls_to_rgb(HUES[m] / 360, l / 100, s / 100)
rgb = np.ones(shape + (3,)); rgb[valid] = [colour(m, u, K2) for m, u in zip(macro, micro)]
mac = np.ones(shape + (3,)); mac[valid] = [colour(m, 1, 3) for m in macro]

lon0, lat0 = transform.c, transform.f
ext = [lon0, lon0 + transform.a * shape[1], lat0 + transform.e * shape[0], lat0]
cty = gpd.read_file(os.path.join(INP, 'kansas_county_bnd.geojson'))
fig, axs = plt.subplots(1, 2, figsize=(12, 3.0))
for ax, img, t in zip(axs, [mac, rgb], [f'{K1} macrozones (rainfall, ETo, GDD)', f'{K2} microzones per macrozone (PAWC, organic matter, clay)']):
    ax.imshow(img, extent=ext, interpolation='nearest'); cty.boundary.plot(ax=ax, color='white', lw=0.3)
    ax.set_aspect(1 / np.cos(np.deg2rad(38.5))); ax.axis('off'); ax.set_title(t, fontsize=10)
plt.tight_layout(); plt.savefig(os.path.join(FIGS, 'zones_default.png'), dpi=300, bbox_inches='tight'); plt.show()

area = np.cos(np.deg2rad(np.broadcast_to((lat0 + transform.e * (np.arange(shape[0]) + 0.5))[:, None], shape)))[valid]
tab = pd.DataFrame(X, columns=keys).assign(zone=code_, w=area).groupby('zone').apply(
    lambda g: pd.Series({'area_%': g.w.sum() / area.sum() * 100, **{k: np.average(g[k], weights=g.w) for k in keys}}))
tab.round(1)

## Export for the app

Writes three files to `data/`:
- `layers.bin.gz`: every layer as uint8 (0–254 between the layer's min and max, 255 = outside Kansas);
- `county_id.bin.gz`: the county of every pixel;
- `meta.json`: grid, layer descriptions and ranges, and simplified county outlines in grid coordinates.

In [ ]:
DESC = {   # role, definition, source: shown in the app's Layers window
    'ppt':  ('Water supply', 'Mean annual precipitation, 1991–2020 normal.', 'PRISM Norm91m (800 m), resampled to the 1-km grid.'),
    'eto':  ('Water demand', 'Mean annual grass reference evapotranspiration (ASCE Penman–Monteith), 1991–2020.', 'gridMET daily (4 km), resampled to the 1-km grid.'),
    'ai':   ('Supply relative to demand', 'Annual rainfall divided by annual ETo. Values below 0.5 are semi-arid, 0.5–0.65 dry subhumid and above 0.65 humid (UNEP classes, which use a different potential ET).', 'Computed from the rainfall and ETo layers.'),
    'awd':  ('Demand not met by rainfall', 'Annual ETo minus annual rainfall. Positive where atmospheric demand exceeds supply.', 'Computed from the rainfall and ETo layers.'),
    'gdd':  ('Thermal time for summer crops', 'Daily (Tmax + Tmin)/2 − 10 °C, with Tmax capped at 30 °C and Tmin floored at 10 °C (the 86/50 °F method), summed per year; mean of 1991–2020.', 'PRISM AN81d daily (4 km), resampled with bilinear interpolation.'),
    'def':  ('Soil water stress', 'Deficit-days: the yearly sum of max(0, 0.5 − FAW), where FAW is the fraction of plant available water in 0–60 cm. Combines how long and how far soil water stayed below half of capacity. Median of the 2002–2025 calendar years.', 'Kansas 1-km soil moisture model.'),
    'sur':  ('Soil water surplus', 'Surplus-days: the yearly sum of max(0, FAW − 0.5). Median of the 2002–2025 calendar years.', 'Kansas 1-km soil moisture model.'),
    'pawc': ('Soil water storage', 'Plant available water capacity of 0–60 cm: water held between −10 kPa and the permanent wilting point.', 'SSURGO texture (CSRL) and the soil limits of the Kansas 1-km soil moisture model.'),
    'som':  ('Soil fertility', 'Soil organic matter in the whole soil profile.', 'SSURGO, gridded by the California Soil Resource Lab (CSRL).'),
    'clay': ('Soil texture', 'Clay content, depth-weighted mean of 0–60 cm (0–25 and 25–50 cm SSURGO layers).', 'SSURGO, gridded by CSRL.'),
    'sand': ('Soil texture', 'Sand content, depth-weighted mean of 0–60 cm (0–25 and 25–50 cm SSURGO layers).', 'SSURGO, gridded by CSRL.'),
}
buf, meta = [], []
for (k, n, u, s, src, _), v in zip(L, X.T):
    lo, hi = float(np.floor(v.min())), float(np.ceil(v.max()))
    if hi - lo < 5: lo, hi = float(np.floor(v.min() * 100) / 100), float(np.ceil(v.max() * 100) / 100)   # small-range layers (aridity index)
    g = np.full(shape, 255, 'uint8'); g[valid] = np.round((v - lo) / (hi - lo) * 254).astype('uint8')
    role, desc, data = DESC[k]
    buf.append(g.tobytes()); meta.append(dict(key=k, name=n, units=u, step=s, source=src, lo=lo, hi=hi, role=role, desc=desc, data=data,
                                              min=float(v.min()), median=float(np.median(v)), max=float(v.max())))
with gzip.open(os.path.join(DATA, 'layers.bin.gz'), 'wb', compresslevel=9) as fh: fh.write(b''.join(buf))

# county of every pixel (1-105 in alphabetical order, 0 = outside); border pixels outside every polygon take the nearest county
from rasterio.features import rasterize
from scipy.ndimage import distance_transform_edt
cty_s = cty.sort_values('NAME').reset_index(drop=True)
cid = rasterize(((g_, i + 1) for i, g_ in enumerate(cty_s.geometry)), out_shape=shape, transform=transform, fill=0, dtype='uint8')
cid = cid[tuple(distance_transform_edt(cid == 0, return_distances=False, return_indices=True))]
cid[~valid] = 0
with gzip.open(os.path.join(DATA, 'county_id.bin.gz'), 'wb', compresslevel=9) as fh: fh.write(cid.tobytes())

simp = cty.to_crs('EPSG:5070').simplify(400).to_crs('EPSG:4326')        # ~400 m tolerance
paths = []
for geom in simp:
    for p in getattr(geom, 'geoms', [geom]):
        xy = np.asarray(p.exterior.coords)
        col = (xy[:, 0] - lon0) / transform.a; row = (xy[:, 1] - lat0) / transform.e
        paths.append('M' + 'L'.join(f'{c:.1f},{r:.1f}' for c, r in zip(col, row)) + 'Z')
lat_c = lat0 + transform.e * shape[0] / 2
json.dump(dict(rows=shape[0], cols=shape[1], transform=list(transform)[:6], crs='EPSG:4326', nodata=255,
               layout='uint8 [layer][row][col], row 0 = north', aspect=float(1 / np.cos(np.deg2rad(lat_c))), layers=meta, counties=paths,
               county_names=cty_s.NAME.tolist(), built=pd.Timestamp.today().strftime('%Y-%m-%d')),
          open(os.path.join(DATA, 'meta.json'), 'w'), separators=(',', ':'))
for f in ('layers.bin.gz', 'county_id.bin.gz', 'meta.json'): print(f, f'{os.path.getsize(os.path.join(DATA, f)) / 1e6:.2f} MB')